<a href="https://colab.research.google.com/github/watermelon-3012/quora-question-pairs-nlp/blob/hanle/bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
torch.cuda.is_available()
import pandas as pd

In [1]:
!apt-get install git --reinstall -y

^C


In [2]:
!pip install transformers datasets seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=46194e36f01ec527aed40ce7644b243273ad9973981fbb2fbd5ad237ece7a2ac
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [5]:
df = pd.read_csv("/content/drive/MyDrive/NLP_Project/train.csv")

In [15]:
df

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0
...,...,...,...,...,...,...
404285,404285,433578,379845,How many keywords are there in the Racket prog...,How many keywords are there in PERL Programmin...,0
404286,404286,18840,155606,Do you believe there is life after death?,Is it true that there is life after death?,1
404287,404287,537928,537929,What is one coin?,What's this coin?,0
404288,404288,537930,537931,What is the approx annual cost of living while...,I am having little hairfall problem but I want...,0


# Data Preprocessing

In [8]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
# Remove N/A data
df = df.dropna(subset=['question1', 'question2'])

In [9]:
lengths = []

for q1, q2 in zip(df['question1'], df['question2']):
    tokens = tokenizer(q1, q2, truncation=False)
    lengths.append(len(tokens['input_ids']))

import numpy as np

print("Mean:", np.mean(lengths))
print("95th percentile:", np.percentile(lengths, 95))
print("Max:", np.max(lengths))

Mean: 30.41209833608303
95th percentile: 54.0
Max: 330


In [19]:
max_seq_len = 330

In [10]:
def tokenize(sample, max_seq_len = 330):
    return tokenizer(
        sample['question1'],
        sample['question2'],
        truncation = True,
        max_length = 128,
    )

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = df['is_duplicate'].values

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = np.array(class_weights, dtype=np.float32)
print(class_weights)

[0.792645  1.3542774]


In [12]:
# Convert dataframe to Hugging Face dataset format
from datasets import Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/404287 [00:00<?, ? examples/s]

In [13]:
# Rename the label column
dataset = dataset.rename_column("is_duplicate", "labels")

In [14]:
# Split
train_dataset = dataset.train_test_split(test_size=0.1)['train']
val_dataset = dataset.train_test_split(test_size=0.1)['test']

In [15]:
from transformers import DataCollatorWithPadding
# Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model

In [16]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
from sklearn.metrics import f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    f1 = f1_score(labels, preds)
    return {"f1": f1}

In [23]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,  # mixed precision for speed
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [21]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
Number of GPUs: 1
GPU name: Tesla T4


In [25]:
# Train model
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.229782,0.176164,0.908020
2,0.164733,0.109086,0.949351
3,0.115016,0.087685,0.962789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=34113, training_loss=0.1923758250422337, metrics={'train_runtime': 4788.6327, 'train_samples_per_second': 227.951, 'train_steps_per_second': 7.124, 'total_flos': 3.793995434971848e+16, 'train_loss': 0.1923758250422337, 'epoch': 3.0})

In [27]:
import os
import json
import pandas as pd
from transformers import Trainer

project_dir = "/content/drive/MyDrive/NLP_Project"
os.makedirs(project_dir, exist_ok=True)  # create folder if not exists

# Save final model & tokenizer
model_dir = os.path.join(project_dir, "saved_model")
trainer.save_model(model_dir)  # model + config
tokenizer.save_pretrained(model_dir)  # tokenizer files

print(f"Model and tokenizer saved to {model_dir}")

# Save evaluation metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
metrics_file = os.path.join(model_dir, "metrics.json")
with open(metrics_file, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Evaluation metrics saved to {metrics_file}")

# Save predictions
preds_output = trainer.predict(val_dataset)
pred_labels = preds_output.predictions.argmax(axis=1)

df_preds = pd.DataFrame({
    "question1": val_dataset["question1"],
    "question2": val_dataset["question2"],
    "true_label": val_dataset["labels"],
    "pred_label": pred_labels
})

preds_file = os.path.join(model_dir, "predictions.csv")
df_preds.to_csv(preds_file, index=False)

print(f"Predictions saved to {preds_file}")

# Clean old checkpoints to save space
results_dir = os.path.join(project_dir, "results")
if os.path.exists(results_dir):
    for folder in os.listdir(results_dir):
        folder_path = os.path.join(results_dir, folder)
        if folder.startswith("checkpoint") and os.path.isdir(folder_path):
            os.system(f"rm -rf {folder_path}")  # delete old checkpoints

print("Old checkpoints cleaned. Storage optimized.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to /content/drive/MyDrive/NLP_Project/saved_model


Evaluation metrics saved to /content/drive/MyDrive/NLP_Project/saved_model/metrics.json
Predictions saved to /content/drive/MyDrive/NLP_Project/saved_model/predictions.csv
Old checkpoints cleaned. Storage optimized.
